# semanticdrift on Google Colab

Runtime → Change runtime type → **GPU**. Keep this tab in the foreground while models download and while `pipeline` runs.

Public clone: `colorfulelina/practice`. Do not put a token in a cell.


In [ ]:
!apt-get -qq update
!apt-get -qq install -y spin gcc g++ make git
!spin -V | head -n 1


In [ ]:
REPO = "colorfulelina/practice"

import os
import subprocess
from pathlib import Path

dest = Path("/content/semanticdrift")
url = f"https://github.com/{REPO}.git"
if not (dest / "src/semanticdrift").is_dir():
    cloned = subprocess.run(
        ["git", "clone", "--depth", "1", url, str(dest)],
        capture_output=True,
        text=True,
    )
    if cloned.returncode != 0:
        raise RuntimeError("git clone failed:\n" + (cloned.stderr or cloned.stdout))

os.chdir(dest)
subprocess.run(["git", "fetch", "--depth", "1", "origin", "main"], check=True)
subprocess.run(["git", "checkout", "-B", "main", "origin/main"], check=True)
!bash scripts/colab_setup.sh


In [ ]:
%cd /content/semanticdrift
!python -m semanticdrift check-promela
!python -m semanticdrift prove-promela


## Language models + pipeline

Install Ollama, pull Qwen then DeepSeek (validate needs both). First run **one** protocol. If that finishes, run all ten. If the session disconnects, start again from Ollama serve (models stay on disk only until the VM is recycled).


In [ ]:
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, urllib.request

subprocess.Popen(["ollama", "serve"])
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/", timeout=1)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ollama serve did not start")

!ollama pull qwen2.5-coder:7b
!ollama pull deepseek-coder-v2:lite
!ollama list


In [ ]:
%cd /content/semanticdrift
!python -m semanticdrift pipeline --name peterson --requirement precise


In [ ]:
%cd /content/semanticdrift
!python -m semanticdrift pipeline --all
!python -m semanticdrift metrics


## 14B parse ablation (not the zero-shot pipeline)

Same few-shot + `spin -a` feedback + keep-last-legal loop as on the laptop, but `qwen2.5-coder:14b`. DeepSeek is not required. This does **not** overwrite `pipeline --all`.

Push this repo to GitHub before cloning, or the CLI will not know `few_shot_verifier_feedback`.

Keep the tab in the foreground. 14B is about 9 GB; one protocol can take several minutes.

In [ ]:
%cd /content/semanticdrift
!apt-get -qq install -y zstd
!command -v ollama >/dev/null || curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, urllib.request

try:
    urllib.request.urlopen("http://127.0.0.1:11434/", timeout=1)
except Exception:
    subprocess.Popen(["ollama", "serve"])
    for _ in range(30):
        try:
            urllib.request.urlopen("http://127.0.0.1:11434/", timeout=1)
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError("ollama serve did not start")

!ollama pull qwen2.5-coder:14b
!ollama list

In [ ]:
%cd /content/semanticdrift
from pathlib import Path
from semanticdrift.promela import check_syntax

names = [
    "peterson",
    "producer_consumer",
    "dining_philosophers",
    "two_phase_commit",
    "tcp_handshake",
]
out = Path("results/baselines/qwen14b")
for name in names:
    !python -m semanticdrift baseline --kind few_shot_verifier_feedback --name {name} --requirement precise --model qwen2.5-coder:14b --out {out}

print("--- spin -a on saved 14B files ---")
for name in names:
    pml = out / f"few_shot_verifier_feedback_{name}_precise.pml"
    ok, detail = check_syntax(pml)
    err = next((ln for ln in detail.splitlines() if "Error:" in ln), "ok" if ok else detail.splitlines()[-1:])
    print(f"{name}: syntax_ok={ok}  {err}")